# Week 1 — What Are We Pricing?
## Overview

This research project asks: **Can we price many variable annuity contracts accurately after simulating only a small, carefully chosen subset?** We will compare ordinary contract distances, clustering, supervised prediction, and one simple way to learn which contract features matter for prices.

This week establishes the quantity every later model will predict. You will work with a small contract table, calculate a guaranteed minimum accumulation benefit (GMAB) payoff, and learn the NumPy and pandas operations needed to study a portfolio. There is no simulation or machine learning model to fit yet.

**By the end, you should be able to:**

- Explain the account value, maturity guarantee, fee, and gross guarantee top-up.
- Translate a payoff into vectorized Python and draw a correctly labelled payoff diagram.
- Distinguish a terminal payoff from a time-zero price.
- Convert between liability per unit of premium and liability in dollars.
- Explain why premium should not be a distance feature for our normalized pricing target.

**Prerequisites:** introductory algebra, the meaning of an average, and basic familiarity with Python variables. We review arrays, tables, and functions as we use them. Prior finance knowledge is not required.

**Concrete outputs:** a checked contract table; a two-panel payoff figure; a normalization table; and a short explanation of our pricing target.

**Time plan:** about 4–5 independent hours, including the reading and exercises, plus a 75-minute meeting. A suggested meeting is 15 minutes on contracts, 25 minutes on the payoff code, 20 minutes on normalization, and 15 minutes on questions and the homework.

**Research scope:** an eight-week research project for three students, followed by two weeks for presentation preparation and rehearsal. The core experiments use a simplified GMAB. More complicated insurance features are topics for discussion, not hidden assumptions in the code.

## 1. A contract is a promise attached to an investment account

A variable annuity links an insurance contract to an investment account. The account can gain or lose value. A **guaranteed minimum accumulation benefit**, or GMAB, promises a floor on the amount available at a specified maturity, subject to the contract conditions.

Our teaching contract has the following fields. Ratios and rates are stored as decimals: `0.20` means 20%, and `0.01` means a continuous annual fee of 1%.

| Symbol | Table column | Meaning | Units |
|---|---|---|---|
| $P$ | `premium` | Initial premium used to normalize the contract | Dollars |
| $a=A_0/P$ | `account_ratio` | Current account value divided by premium | Per unit of premium |
| $g=G/P$ | `guarantee_ratio` | Maturity guarantee divided by premium | Per unit of premium |
| $T$ | `maturity` | Years remaining until the guarantee is paid | Years |
| $\sigma$ | `volatility` | Annual volatility of the investment account | Annual model parameter |
| $\phi$ | `fee` | Specified continuous proportional fee deducted from the account | Per year |
| — | `contract_id` | Identifier used to track the same contract | Identifier, not a model feature |

The pricing interest rate is fixed at $r=0.03$ in all core notebooks. We vary the other contract characteristics so the portfolio is heterogeneous. A current account ratio need not equal one: the account may have changed since the premium was paid.

**Our precise target:** the time-zero value of the insurer's **gross GMAB top-up**, assuming the contract survives to maturity. No survival-probability weighting is applied. We do not model deaths, lapses, withdrawals, ratchets, taxes, or fee revenue. We hold the fee fixed; finding a fair fee is a different problem introduced in one of the readings.

The following table is deliberately small. A research portfolio will contain many more rows, but the same column meanings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

pd.set_option("display.precision", 5)
r = 0.03

contracts = pd.DataFrame({
    "contract_id": [101, 102, 103],
    "premium": [20000.0, 35000.0, 50000.0],
    "account_ratio": [1.00, 0.90, 1.10],
    "guarantee_ratio": [1.00, 1.10, 1.00],
    "maturity": [10.0, 15.0, 8.0],
    "volatility": [0.20, 0.25, 0.15],
    "fee": [0.01, 0.015, 0.005],
})

display(contracts)
print(f"Rows: {len(contracts)}; columns: {len(contracts.columns)}")
print("All premiums positive:", bool((contracts["premium"] > 0).all()))
print("All maturities positive:", bool((contracts["maturity"] > 0).all()))
print("Contract identifiers unique:", bool(contracts["contract_id"].is_unique))

### Python lab: a row, a column, and a derived column

A pandas `DataFrame` is a labelled table. `contracts["premium"]` selects one column; `contracts.iloc[0]` selects the first row by position. A column can be converted into a NumPy array using `.to_numpy()` when we want numerical array operations.

Multiplying two columns acts row by row. For example, $A_0=P a$ gives each contract's current account value. Do not confuse the row's position with `contract_id`: the identifier should follow the contract if we later shuffle or split the table.

In [ ]:
contract_summary = contracts.copy()
contract_summary["current_account_dollars"] = (
    contract_summary["premium"] * contract_summary["account_ratio"]
)
contract_summary["guarantee_dollars"] = (
    contract_summary["premium"] * contract_summary["guarantee_ratio"]
)

display(contract_summary[["contract_id", "premium", "current_account_dollars",
                          "guarantee_dollars", "maturity"]])

first_contract = contracts.iloc[0]
premium_array = contracts["premium"].to_numpy()
print("First contract ID:", int(first_contract["contract_id"]))
print("Premium array:", premium_array)
print("Total premium in this small table:", premium_array.sum())

## 2. Derive the guarantee payoff before writing a pricing model

Let $A_T$ be the account value at maturity and let $G=P g$ be the promised maturity floor. The insurer pays the shortfall only when the account is below the floor:

$$H_T=\max(G-A_T,0)=(G-A_T)^+.$$

This gives two cases:

$$H_T=\begin{cases}G-A_T,&A_T<G,\\0,&A_T\geq G.\end{cases}$$

The total maturity payment from the account and this guarantee is

$$A_T+H_T=\max(A_T,G).$$

Our model prices $H_T$, the top-up. The entire maturity payment is a different cash flow. For example, with $P=\$20{,}000$ and $g=1$, an account ending at $\$16{,}000$ needs a $\$4{,}000$ top-up; an account ending at $\$24{,}000$ needs no top-up.

In NumPy, `np.maximum(x, 0.0)` performs the positive-part operation element by element. An array lets us compute all scenarios at once.

In [ ]:
P = float(first_contract["premium"])
g = float(first_contract["guarantee_ratio"])
G = P * g

terminal_accounts = np.array([0.0, 12000.0, 16000.0, 20000.0, 24000.0, 32000.0])
topups = np.maximum(G - terminal_accounts, 0.0)
total_payments = terminal_accounts + topups

scenario_table = pd.DataFrame({
    "terminal_account_dollars": terminal_accounts,
    "guarantee_dollars": G,
    "topup_dollars": topups,
    "total_maturity_payment_dollars": total_payments,
})
display(scenario_table)

# A model identity is more informative than checking one hand-picked number.
assert np.allclose(total_payments, np.maximum(terminal_accounts, G))
assert np.all(topups >= 0.0)
assert np.all(topups <= G)

### A function makes the cash-flow definition reusable

A Python function gives a name to a calculation. The function below accepts either one number or an array. Its arguments are all in dollars, so its result is in dollars too. The guarantee is paid once at maturity; there are no intermediate withdrawals in this model.

In [ ]:
def gmab_topup_dollars(terminal_account, guarantee):
    """Gross maturity top-up; account and guarantee use the same currency units."""
    return np.maximum(np.asarray(guarantee) - np.asarray(terminal_account), 0.0)

account_grid = np.linspace(0.0, 1.8 * P, 181)
payoff_grid = gmab_topup_dollars(account_grid, G)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(account_grid, payoff_grid, linewidth=2, label="GMAB top-up")
axes[0].axvline(G, color="gray", linestyle="--", label="Guarantee level")
axes[0].set(xlabel="Account value at maturity ($)", ylabel="Insurer top-up at maturity ($)",
            title="The cash flow our project prices")
axes[0].legend()

axes[1].plot(account_grid, account_grid + payoff_grid, linewidth=2,
             label="Account plus GMAB top-up")
axes[1].plot(account_grid, account_grid, linestyle="--", color="gray", label="Account alone")
axes[1].set(xlabel="Account value at maturity ($)", ylabel="Total maturity payment ($)",
            title="The full maturity payment is a different target")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

**Interpretation checkpoint.** Identify the kink in each graph. Why does the top-up decrease as the terminal account increases? Why is a zero top-up compatible with the policyholder receiving a large maturity payment? Write two sentences before moving on.

## 3. Normalize first: separate contract shape from contract size

Write $X_T=A_T/P$. Dividing the top-up by premium gives

$$\frac{H_T}{P}=\max(g-X_T,0).$$

If we multiply $P$, $A_0$, and $G$ by the same positive factor, then $a=A_0/P$ and $g=G/P$ stay unchanged. Under our later account model, the distribution of $X_T$ is also unchanged. Consequently the **price per unit of premium is unchanged**, while the dollar price grows in proportion to $P$.

This matters for contract similarity. Two otherwise identical contracts with premiums of $\$20{,}000$ and $\$80{,}000$ have the same normalized target. A distance that treats their $\$60{,}000$ premium difference as evidence of different contract shapes would be misleading for that target.

Our feature vector will therefore be

$$x=(a,g,T,\sigma,\phi).$$

Premium is retained for converting predictions to dollars and computing total portfolio liability. It is excluded from the normalized-price model. This simplification relies on the proportional structure of our teaching model; fixed dollar charges or other product rules could break it.

In [ ]:
# One common scenario: the account ends at 80% of premium.
premiums = np.array([20000.0, 40000.0, 80000.0])
terminal_account_ratio = 0.80
guarantee_ratio = 1.00

normalization_table = pd.DataFrame({
    "premium_dollars": premiums,
    "terminal_account_dollars": premiums * terminal_account_ratio,
    "guarantee_dollars": premiums * guarantee_ratio,
})
normalization_table["topup_dollars"] = gmab_topup_dollars(
    normalization_table["terminal_account_dollars"],
    normalization_table["guarantee_dollars"],
)
normalization_table["topup_per_premium"] = (
    normalization_table["topup_dollars"] / normalization_table["premium_dollars"]
)
display(normalization_table)

assert np.allclose(normalization_table["topup_per_premium"], 0.20)
FEATURES = ["account_ratio", "guarantee_ratio", "maturity", "volatility", "fee"]
print("Later prediction features:", FEATURES)

### Percentages, percentage points, and dollars

A normalized liability of `0.04` means 4% of premium. For a $\$25{,}000$ contract it corresponds to $\$1{,}000$. If the prediction is `0.045`, its error is `0.005` per premium: **0.5 percentage points of premium**, or $\$125$ for that contract. It is not a relative error of 0.5% in the liability; relative to a liability of `0.04`, it is a 12.5% error.

Later plots will use percentage points of premium for per-contract pricing errors. We will explicitly label portfolio relative errors separately.

In [ ]:
example_premium = 25000.0
example_true_value = 0.04
example_prediction = 0.045
example_error = example_prediction - example_true_value

units_table = pd.DataFrame({
    "quantity": ["Illustrative normalized price", "Illustrative prediction error"],
    "per_premium": [example_true_value, example_error],
    "percent_or_percentage_points_of_premium": [100 * example_true_value,
                                                  100 * example_error],
    "dollars": [example_premium * example_true_value, example_premium * example_error],
})
display(units_table)
print("Illustration only: these prices were chosen to explain units, not estimated.")

## 4. A payoff is not yet a price

A payoff depends on what happens at maturity. A time-zero price averages discounted future payoffs under a specified valuation model. For our target,

$$y(x)=\mathbb E_Q\!\left[e^{-rT}\max(g-X_T,0)\right], \qquad L=P\,y(x).$$

Here $Q$ denotes the **risk-neutral valuation law**, introduced next week. It is not simply a collection of equally likely market stories chosen by hand. $y$ is the price per unit of premium and $L$ is the price in dollars.

Today we can calculate the discounted payoff for any given scenario:

$$Y=e^{-rT}\max(g-X_T,0).$$

Discounting recognizes that a payment at time $T$ is valued at time zero using the model interest rate. An average of arbitrarily chosen scenarios is useful as a coding exercise, but it becomes a Monte Carlo price estimate only when the scenarios are actually drawn from the stated valuation law.

The continuous fee $\phi$ will reduce account growth in our model. Holding a market scenario fixed, a larger fee therefore lowers the terminal account and **increases or leaves unchanged the gross guarantee top-up**. The full account-plus-guarantee payment is a different target. It is a supplied contract input. Our gross top-up target does not subtract future fee income. In the mentor's fee-setting manuscript, the unknown fee is instead chosen to satisfy a pricing equation; that is a separate mathematical task.

In [ ]:
T = float(first_contract["maturity"])
scenario_ratios = np.array([0.5, 0.8, 1.0, 1.2, 1.5])
normalized_topups = np.maximum(g - scenario_ratios, 0.0)
discounted_normalized_payoffs = np.exp(-r * T) * normalized_topups

discount_table = pd.DataFrame({
    "terminal_account_per_premium": scenario_ratios,
    "topup_at_maturity_per_premium": normalized_topups,
    "discounted_topup_per_premium": discounted_normalized_payoffs,
    "discounted_topup_dollars": P * discounted_normalized_payoffs,
})
display(discount_table)
print(f"Discount factor for T={T:g} years at r={r:.2%}: {np.exp(-r*T):.5f}")
print("These hand-chosen scenarios are not a Monte Carlo price sample.")

## 5. Turn the pricing question into a research question

The complete simulation baseline will generate many random account outcomes for **every** contract, calculate discounted top-ups, and average them. That approach gives us a transparent benchmark but can require a great deal of simulation in richer products.

Our alternative will use three stages:

1. **Compressor:** choose a small set of representative contracts.
2. **Simulator:** estimate the representatives' prices by Monte Carlo.
3. **Predictor:** use their inputs and simulated prices to predict prices for other contracts.

An ordinary nearest-neighbor predictor assumes that nearby input features imply similar prices. A regression model learns a mapping from features to prices. A learned distance changes which features count most when choosing neighbors. These are related but different approaches, and our experiments will distinguish them.

The final research question is deliberately measurable: **At an equal simulation budget, which representation and predictor gives the best accuracy on contracts that were held out in advance, and what computational work does it require?** We will not promise a particular winning model or speedup.

Because our GMAB has a simple analytical price, it is a useful teaching test bed. The analytical formula will be used to audit our simulator and, after the final benchmark is frozen, to inspect test accuracy. It will not supply training or validation labels.

## Reading and discussion

Read for the question beside each source; you are not expected to master the whole source.

| Source | This week's task | Discussion question |
|---|---|---|
| Mary Hardy (2003), [*Investment Guarantees: Modeling and Risk Management for Equity-Linked Life Insurance*](https://www.wiley-vch.de/en/areas-interest/finance-economics-law/finance-investments-13fi/investments-securities-13fi3/investment-guarantees-978-0-471-39290-3). Wiley. Use a library or mentor-provided copy. | Read the opening discussion of investment guarantees and the passages introducing maturity guarantees; spend about 20 minutes. | What financial uncertainty creates the need for a guarantee, and which real contract features have we omitted? |
| Daniel Bauer, Alexander Kling, and Jochen Russ (2008), [*A Universal Pricing Framework for Guaranteed Minimum Benefits in Variable Annuities*](https://doi.org/10.2143/AST.38.2.2033356), *ASTIN Bulletin*, 38(2), 621–651. | Read the abstract, introduction, and the definitions of the main guarantee types; spend about 20 minutes. | How does a GMAB differ from a death benefit and a withdrawal guarantee? Which one does this notebook model? |

**Submit three short reading answers:** (1) define a GMAB in your own words; (2) name two omitted product features and explain what extra information each would require; (3) explain why the policyholder's total maturity payment and the insurer's guarantee top-up have different pricing targets. Cite the source you used. A concise paragraph for each is sufficient.

## Coding homework 1 — Change the guarantee, keep the account scenarios fixed

Use the starter below to draw top-up curves for guarantee ratios $g\in\{0.9,1.0,1.1\}$ on the **same** normalized terminal-account grid. Label every curve, both axes, and the units. Then convert the three curves to dollars for a $\$30{,}000$ premium.

Report the top-up at account ratios `0.8`, `1.0`, and `1.2` for each guarantee. Explain which parts of the graph change when the guarantee changes and which parts change when only premium changes.

The starter runs as supplied for one guarantee; extend it rather than replacing the payoff definition.

In [ ]:
# Homework starter: add the other guarantee ratios and a dollar-valued comparison.
hw_account_ratios = np.linspace(0.0, 1.6, 161)
hw_guarantees = [1.0]
hw_premium = 30000.0

fig, ax = plt.subplots(figsize=(6, 4))
for hw_g in hw_guarantees:
    hw_topup_ratio = np.maximum(hw_g - hw_account_ratios, 0.0)
    ax.plot(hw_account_ratios, hw_topup_ratio, label=f"g = {hw_g:.1f}")
ax.set(xlabel="Terminal account / premium", ylabel="Top-up / premium",
       title="Homework: compare guarantee levels")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Coding homework 2 — Build a normalization check

Create four contracts with the same $a,g,T,\sigma,\phi$ and different premiums. For one shared vector of normalized terminal-account scenarios, calculate the discounted top-up both ways:

- Directly in dollars: $e^{-rT}\max(Pg-PX_T,0)$.
- In normalized units and then rescaled: $P e^{-rT}\max(g-X_T,0)$.

Show that these methods agree numerically for every contract and scenario using `np.allclose`. Produce one table with the premium, one scenario's dollar payoff, and its normalized payoff. Explain why `premium` is useful for reporting but unnecessary as a normalized-price feature.

**Extension, optional:** Why could a fixed dollar charge undermine this scaling argument? You do not need to implement a new insurance product.

In [ ]:
# Homework starter: complete the two-route comparison for all scenarios.
hw_premiums = np.array([10000.0, 25000.0, 50000.0, 80000.0])
hw_scenarios = np.array([0.65, 0.95, 1.25])
hw_g = 1.0
hw_T = 10.0

# Broadcasting: shape (4, 1) multiplied by shape (1, 3) gives a (4, 3) table.
hw_terminal_dollars = hw_premiums[:, None] * hw_scenarios[None, :]
hw_guarantee_dollars = hw_premiums[:, None] * hw_g
hw_direct_discounted_payoffs = (
    np.exp(-r * hw_T) * np.maximum(hw_guarantee_dollars - hw_terminal_dollars, 0.0)
)
display(pd.DataFrame(hw_direct_discounted_payoffs,
                     index=hw_premiums,
                     columns=[f"account_ratio_{x:.2f}" for x in hw_scenarios]))
# Add the normalized route, an equality check, and your reporting table here.

## Weekly deliverable and checklist

Submit your completed notebook with your name, the two homework outputs, and the three reading answers. Add a 150–200 word explanation of what the project will predict and why it may be possible to simulate fewer contracts.

- [ ] I can explain every column in the contract table and its units.
- [ ] I can derive and code the GMAB top-up, and distinguish it from the total payment.
- [ ] My figures have titles, legends where needed, and correct axis units.
- [ ] I can explain the difference between a payoff, a discounted payoff, and a price.
- [ ] I can convert normalized errors into dollars and percentage points of premium.
- [ ] I understand that the fee is supplied, and that this is not a fair-fee solver.
- [ ] I restarted the kernel and ran every cell from top to bottom.

**Three-student meeting roles:** one person explains the insurance terms, one walks through the code, and one checks units and assumptions. Rotate roles next week. Every student completes their own notebook.